We used Gemini and Claude to help with the coding

In [1]:
import pandas as pd
import numpy as np

In [185]:
df = pd.read_csv("master_data_2020.csv")
df_raw = pd.read_csv("master_data_2020.csv")
df = df[~df["presvote20post"].isna()]

# ==========================================
# 1. DEFINE SOURCE COLUMNS
# ==========================================
treatment = "untrustworthy_flag"
raw_vote_choice_col = "presvote20post"

# ==========================================
# 2. OVERWRITE & CLEAN TARGET OUTCOMES FROM TEXT
# ==========================================
# Ensure we capture text responses and handle any trailing whitespaces cleanly
vote_series = df[raw_vote_choice_col].astype(str).str.strip()

# 1. Fix Biden Vote (1 if explicitly voted for Biden, 0 otherwise)
df["voted_biden_2020"] = np.where(vote_series == "Joe Biden", 1, 0)

# 2. Fix Trump Vote (1 if explicitly voted for Trump, 0 otherwise)
df["voted_trump_2020"] = np.where(vote_series == "Donald Trump", 1, 0)

# 3. Create Turnout (0 if they didn't vote or skipped, 1 if they cast a ballot)
# Based on your data, non-voters show up as "Did not vote for President" or missing strings
df["turnout_2020_binary"] = np.where(
    vote_series.str.contains("did not vote|skipped|__na__", case=False, na=True),
    0,
    1
)

# Structure our clean target dictionary for the loop
outcomes = {
    "2020 Turnout": "turnout_2020_binary",
    "Voted for Biden": "voted_biden_2020",
    "Voted for Trump": "voted_trump_2020",
}

# ==========================================
# 3. COMPREHENSIVE FEATURE ENGINEERING (CONDENSED & STREAMLINED)
# ==========================================
# Process Digital Literacy Index
tf_cols = ["tf_adv", "tf_pdf", "tf_spy", "tf_wiki", "tf_cache", "tf_phishing"]
df_tf = df[tf_cols].map(
    lambda val: int(str(val).strip().split()[0])
    if pd.notna(val) and str(val).strip().split()[0].isdigit()
    else np.nan
)
df["digital_literacy_index"] = df_tf.mean(axis=1)

# Linearized Political Mappings
df["ideo5_linear"] = df["ideo5"].map({"Very liberal": 1, "Liberal": 2, "Moderate": 3, "Conservative": 4, "Very conservative": 5})
df["pid7_linear"] = df["pid7"].map({"Strong Democrat": 1, "Not very strong Democrat": 2, "Lean Democrat": 3, "Independent": 4, "Lean Republican": 5, "Not very strong Republican": 6, "Strong Republican": 7})

# Linearized Social Media Scale
socmed_mapping = {
    "Less than 10 minutes per day": 1, "10–30 minutes per day": 2, "31–60 minutes per day": 3,
    "1–2 hours per day": 4, "2–3 hours per day": 5, "More than 3 hours per day": 6
}
df["socmed_use_linear"] = df["socmed_use"].astype(str).str.strip().map(socmed_mapping)

# --- NEW: Condense News Interest into a Linear Scale ---
newsint_mapping = {
    "Hardly at all": 1,
    "Only now and then": 2,
    "Some of the time": 3,
    "Most of the time": 4
}
df["newsint_linear"] = df["newsint"].astype(str).str.strip().map(newsint_mapping)

# --- NEW: Condense Internet Use Frequency into a Linear Scale ---
intuse_mapping = {
    "Less often": 1,
    "About once a day": 3,
    "Several times a day": 4,
    "A few times a week": 2  # Catching common top-tier survey labels if present
}
# Fallback to handle "Several times a day" as the top option if "Almost constantly" isn't used
df["intuse_linear"] = df["intuse"].astype(str).str.strip().map(intuse_mapping).fillna(3)

# --- NEW: Condense 2016 Vote into Trump, Clinton, and Other ---
# Modified 2016 vote processing function
def condense_2016_vote_fixed(val):
    val_str = str(val).strip()
    if val_str in ["Hillary Clinton", "Donald Trump"]:
        return val_str
    # Explicitly group non-voters and missing responses into a valid category string
    elif pd.isna(val) or val_str in ["__NA__", "nan", "Did not vote", "Did not vote for President"]:
        return "Did Not Vote"
    else:
        return "Other"

df["presvote16post_condensed"] = df["presvote16post"].apply(condense_2016_vote_fixed)

# REMOVE 'turnout16' from your categorical_covariates list!
# 'presvote16post_condensed' now handles turnout implicitly.
categorical_covariates = [
    "race4", 
    "educ4", 
    "region", 
    "presvote16post_condensed" # Will generate dummies for Clinton, Other, and Did Not Vote. 
]
# Gather all our dense numeric/linear features
linear_features = [
    "ideo5_linear", 
    "pid7_linear", 
    "socmed_use_linear", 
    "newsint_linear", 
    "intuse_linear"
]

# ==========================================
# 4. DESIGN MATRIX GENERATION (THE COVARIATES)
# ==========================================
core_cols = [treatment] + list(outcomes.values()) + ["age", "female"]
engineered_continuous = ["digital_literacy_index"]


# Clean complete-case alignment
all_processed_cols = core_cols + engineered_continuous + linear_features + categorical_covariates
df_clean = df[all_processed_cols].dropna().copy()

# Build the final compact X Matrix
# 1. Generate all dummies with drop_first=False so we can manually control the baselines
X_matrix = pd.get_dummies(
    df_clean[["age", "female"] + engineered_continuous + linear_features + categorical_covariates],
    columns=categorical_covariates,
    drop_first=False,
    dtype=int,
)

# 2. Define exactly one baseline reference column to drop from each categorical group
baselines_to_drop = [
    'presvote16post_condensed_Donald Trump',  # 2016 Vote baseline
    'race4_White',                            # Race baseline
    'educ4_College grad',                     # Education baseline
    'region_Midwest',                         # Region baseline
]

# 3. Drop them safely from the matrix
X_matrix = X_matrix.drop(columns=baselines_to_drop, errors='ignore')

# ==========================================
# 5. FINAL AIPW INPUT EXTRACTION
# ==========================================
W = df_clean[treatment].astype(int).values
Y_turnout = df_clean[outcomes["2020 Turnout"]].astype(int).values
Y_biden = df_clean[outcomes["Voted for Biden"]].astype(int).values
Y_trump = df_clean[outcomes["Voted for Trump"]].astype(int).values

print("--- PIPELINE VERIFICATION ---")
print(f"Total valid sample size (N): {len(df_clean)}")
print(f"Total covariates in X_matrix: {X_matrix.shape[1]}")
print(f"Mean 2020 Turnout rate: {Y_turnout.mean():.2%}")
print(f"Mean unconditional Biden support: {Y_biden.mean():.2%}")
print(f"Mean unconditional Trump support: {Y_trump.mean():.2%}")

raw_post_weights = df.loc[X_matrix.index, "weight_post"].fillna(1.0).values

# 2. Dynamically calculate the 5th and 95th percentiles
lower_bound = np.percentile(raw_post_weights, 5)
upper_bound = np.percentile(raw_post_weights, 95)

# 3. Perform the symmetric percentile clip (Winsorization)
trimmed_post_weights = np.clip(raw_post_weights, lower_bound, upper_bound)

# 4. Re-normalize so the final vector averages exactly 1.0
survey_weights = trimmed_post_weights / np.mean(trimmed_post_weights)

--- PIPELINE VERIFICATION ---
Total valid sample size (N): 956
Total covariates in X_matrix: 20
Mean 2020 Turnout rate: 97.38%
Mean unconditional Biden support: 59.73%
Mean unconditional Trump support: 35.46%


In [186]:
print("--- SURVEY WEIGHT DIAGNOSTICS ---")
print(f"Min Weight:     {survey_weights.min():.4f}")
print(f"Max Weight:     {survey_weights.max():.4f}")
print(f"Mean Weight:    {survey_weights.mean():.4f}")
print(f"Std Dev Weight: {survey_weights.std():.4f}")

# Check the ratio of max to mean
print(f"Max/Mean Ratio: {survey_weights.max() / survey_weights.mean():.2f}")

--- SURVEY WEIGHT DIAGNOSTICS ---
Min Weight:     0.1159
Max Weight:     3.3223
Mean Weight:    1.0000
Std Dev Weight: 0.8259
Max/Mean Ratio: 3.32


In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.linear_model import LogisticRegression

# print("--- COMPUTING PANEL ATTRITION PROPENSITY SCORES & REPAIRING WEIGHTS ---")

# # 1. Define survival indicator based on index matches with your final clean sample
# df_raw["survived_preprocessing"] = df_raw.index.isin(df_clean.index).astype(int)

# # 2. Replicate the streamlined mappings onto df_raw to capture baseline profiles of dropouts
# df_raw["ideo5_linear"] = df_raw["ideo5"].map({"Very liberal": 1, "Liberal": 2, "Moderate": 3, "Conservative": 4, "Very conservative": 5})
# df_raw["pid7_linear"] = df_raw["pid7"].map({"Strong Democrat": 1, "Not very strong Democrat": 2, "Lean Democrat": 3, "Independent": 4, "Lean Republican": 5, "Not very strong Republican": 6, "Strong Republican": 7})

# socmed_mapping = {
#     "Less than 10 minutes per day": 1, "10–30 minutes per day": 2, "31–60 minutes per day": 3,
#     "1–2 hours per day": 4, "2–3 hours per day": 5, "More than 3 hours per day": 6
# }
# df_raw["socmed_use_linear"] = df_raw["socmed_use"].astype(str).str.strip().map(socmed_mapping)

# newsint_mapping = {"Hardly at all": 1, "Only now and then": 2, "Some of the time": 3, "Most of the time": 4}
# df_raw["newsint_linear"] = df_raw["newsint"].astype(str).str.strip().map(newsint_mapping)

# intuse_mapping = {"Less often": 1, "About once a day": 3, "Several times a day": 4, "A few times a week": 2}
# df_raw["intuse_linear"] = df_raw["intuse"].astype(str).str.strip().map(intuse_mapping).fillna(3)

# # 3. Establish the foundational attrition feature space
# attrition_features = ["age", "female", "ideo5_linear", "pid7_linear", "socmed_use_linear", "newsint_linear", "intuse_linear"]

# # Median-impute missing baseline spaces in df_raw ONLY to prevent the attrition model from losing rows
# df_attrition_matrix = df_raw[attrition_features].copy()
# for col in attrition_features:
#     df_attrition_matrix[col] = df_attrition_matrix[col].fillna(df_attrition_matrix[col].median())

# X_attrition = df_attrition_matrix.values
# y_attrition = df_raw["survived_preprocessing"].values

# # 4. Fit Response Propensity Model
# attrition_clf = LogisticRegression(max_iter=2000, random_state=42)
# attrition_clf.fit(X_attrition, y_attrition)

# # Extract predicted probability of remaining in the study
# prob_survival = attrition_clf.predict_proba(X_attrition)[:, 1]
# df_raw["prob_survival"] = prob_survival

# # 5. Map survival adjustments back onto the clean survivors via index alignment
# survival_prob_clean = df_raw.loc[df_clean.index, "prob_survival"].values
# original_weights_clean = df_raw.loc[df_clean.index, "weight"].values

# # Apply the Inverse Probability of Attrition correction math
# # Weight_repaired = Weight_original * (1 / P(Survival))
# adjusted_weights = original_weights_clean * (1.0 / survival_prob_clean)

# # Re-normalize weights so they average out to exactly 1 across your active sample
# survey_weights = adjusted_weights / np.mean(adjusted_weights)

# print("\n[Weight Optimization Diagnostics]")
# print(f"  Initial raw rows:                      {len(df_raw)}")
# print(f"  Surviving clean rows:                  {len(df_clean)}")
# print(f"  Original weights mean in clean sample: {np.mean(original_weights_clean):.4f}")
# print(f"  Repaired weights mean (re-centered):  {np.mean(survey_weights):.4f}")

# # 6. CHARACTERIZE THE TREATED (Generate profile metrics for your slide deck)
# df_profile = df_clean.copy()
# df_profile["untrustworthy_flag"] = W

# profile_cols = ["age", "female"] + linear_features
# treated_means = df_profile[df_profile["untrustworthy_flag"] == 1][profile_cols].mean()
# control_means = df_profile[df_profile["untrustworthy_flag"] == 0][profile_cols].mean()

# print("\n[Presentation Insight: Descriptive Profile of the Treated vs Control]")
# print(f"{'Covariate Feature':<25} | {'Treated Mean (W=1)':<20} | {'Control Mean (W=0)':<20}")
# print("-" * 72)
# for feat in profile_cols:
#     print(f"{feat:<25} | {treated_means[feat]:<20.4f} | {control_means[feat]:.4f}")

--- COMPUTING PANEL ATTRITION PROPENSITY SCORES & REPAIRING WEIGHTS ---

[Weight Optimization Diagnostics]
  Initial raw rows:                      1151
  Surviving clean rows:                  956
  Original weights mean in clean sample: 0.9740
  Repaired weights mean (re-centered):  1.0000

[Presentation Insight: Descriptive Profile of the Treated vs Control]
Covariate Feature         | Treated Mean (W=1)   | Control Mean (W=0)  
------------------------------------------------------------------------
age                       | 58.7910              | 54.7891
female                    | 0.5145               | 0.5473
ideo5_linear              | 3.1190               | 2.7690
pid7_linear               | 4.1061               | 3.1194
socmed_use_linear         | 2.6592               | 2.4729
newsint_linear            | 3.6592               | 3.4698
intuse_linear             | 3.9003               | 3.8589


In [ ]:
# # Calculate the raw inverse-probability adjusted weights
# adjusted_weights = original_weights_clean * (1.0 / survival_prob_clean)

# # --- WINZORIZATION: Cap the top 5% of extreme weights ---
# upper_bound = np.percentile(adjusted_weights, 95)
# adjusted_weights_trimmed = np.clip(adjusted_weights, None, upper_bound)

# # Re-normalize the trimmed weights so they average out to exactly 1
# survey_weights = adjusted_weights_trimmed / np.mean(adjusted_weights_trimmed)

# print(f"Trimming complete. New Max Weight is capped at: {survey_weights.max():.4f}")

Trimming complete. New Max Weight is capped at: 3.8775


In [ ]:
# import numpy as np
# import pandas as pd
# import scipy.stats as stats
# import statsmodels.api as sm

# # 1. statsmodels requires an explicit intercept column
# X_with_constant = sm.add_constant(X_matrix)

# # Pack outcomes into a dictionary matching our extracted variables
# outcome_data = {
#     "Voted for Biden": Y_biden,
#     "Voted for Trump": Y_trump
# }

# print("--- RUNNING LOGISTIC AIPW ESTIMATION (STATSMODELS) ---")

# for name, Y in outcome_data.items():
#     try:
#         # ----------------------------------------------------
#         # STEP 1: Propensity Score Model e(X) via pure MLE
#         # ----------------------------------------------------
#         prop_model = sm.Logit(W, X_with_constant).fit(disp=0, maxiter=1000)
#         e_hat = prop_model.predict(X_with_constant)
        
#         # Clip propensity scores to protect against extreme weights
#         e_hat = np.clip(e_hat, 0.05, 0.95)
        
#         # ----------------------------------------------------
#         # STEP 2: Outcome Models mu_1(X) and mu_0(X)
#         # ----------------------------------------------------
#         # Fit model only on the treated units (W == 1)
#         out_model_1 = sm.Logit(Y[W == 1], X_with_constant.iloc[W == 1]).fit(disp=0, maxiter=1000)
#         mu_1_hat = out_model_1.predict(X_with_constant)
        
#         # Fit model only on the control units (W == 0)
#         out_model_0 = sm.Logit(Y[W == 0], X_with_constant.iloc[W == 0]).fit(disp=0, maxiter=1000)
#         mu_0_hat = out_model_0.predict(X_with_constant)
        
#         # ----------------------------------------------------
#         # STEP 3: Compute AIPW Scores & Asymptotic Variance
#         # ----------------------------------------------------
#         N = len(Y)
        
#         base_diff = mu_1_hat - mu_0_hat
#         treated_correction = (W * (Y - mu_1_hat)) / e_hat
#         control_correction = ((1 - W) * (Y - mu_0_hat)) / (1 - e_hat)
        
#         # Generate final point-level scores
#         aipw_scores = base_diff + treated_correction - control_correction
        
#         # Unbiased Average Treatment Effect (ATE)
#         ate = np.mean(aipw_scores)
        
#         # Analytical standard errors derived from the empirical influence curve
#         influence_curve = aipw_scores - ate
#         variance = np.var(influence_curve, ddof=1) / N
#         std_error = np.sqrt(variance)
        
#         # Compute Wald Z-Test statistics
#         z_stat = ate / std_error
#         p_val = 2 * (1 - stats.norm.cdf(np.abs(z_stat)))
        
#         # Confidence Intervals (95%)
#         ci_lower = ate - (1.96 * std_error)
#         ci_upper = ate + (1.96 * std_error)
        
#         # ----------------------------------------------------
#         # STEP 4: Print Clean Estimates
#         # ----------------------------------------------------
#         print(f"\nOutcome: {name}")
#         print(f"  ATE: {ate:+.4f} ({ate*100:+.2f} percentage points)")
#         print(f"  Std Error: {std_error:.4f}")
#         print(f"  95% CI:   [{ci_lower:+.4f}, {ci_upper:+.4f}]")
#         print(f"  p-value:   {p_val:.4f}")
        
#     except Exception as e:
#         print(f"\nModel for '{name}' failed to converge or encountered an error:")
#         print(f"  Error details: {e}")

--- RUNNING LOGISTIC AIPW ESTIMATION (STATSMODELS) ---

Outcome: Voted for Biden
  ATE: -0.0286 (-2.86 percentage points)
  Std Error: 0.0149
  95% CI:   [-0.0578, +0.0006]
  p-value:   0.0547

Outcome: Voted for Trump
  ATE: +0.0144 (+1.44 percentage points)
  Std Error: 0.0147
  95% CI:   [-0.0144, +0.0432]
  p-value:   0.3279


In [ ]:
# import numpy as np
# import pandas as pd
# import scipy.stats as stats
# from sklearn.ensemble import RandomForestClassifier

# # Use our clean master X_matrix values
# X = X_matrix.values
# N = len(W)

# candidate_outcomes = {"Voted for Biden": Y_biden, "Voted for Trump": Y_trump}

# # Native NumPy K-Fold generation
# np.random.seed(42)
# indices = np.arange(N)
# np.random.shuffle(indices)

# num_folds = 5
# folds = np.array_split(indices, num_folds)

# print("--- RUNNING LIGHTWEIGHT RANDOM FOREST AIPW ---")

# for name, Y in candidate_outcomes.items():
#     # Placeholders for out-of-fold probability predictions
#     e_hat = np.zeros(N)
#     mu_1_hat = np.zeros(N)
#     mu_0_hat = np.zeros(N)

#     for f in range(num_folds):
#         val_idx = folds[f]
#         train_idx = np.setdiff1d(indices, val_idx)

#         X_train, X_val = X[train_idx], X[val_idx]
#         W_train, W_val = W[train_idx], W[val_idx]
#         Y_train, Y_val = Y[train_idx], Y[val_idx]

#         # ----------------------------------------------------
#         # STEP 1: Lightweight Propensity Score Model e(X)
#         # ----------------------------------------------------
#         # Max depth = 2 prevents hyper-partisans from getting 0.99 or 0.01 propensities
#         prop_model = RandomForestClassifier(
#             n_estimators=100, 
#             max_depth=10, 
#             min_samples_leaf=10,
#             random_state=42, 
#             n_jobs=-1
#         )
#         prop_model.fit(X_train, W_train)
#         e_hat[val_idx] = prop_model.predict_proba(X_val)[:, 1]

#         # ----------------------------------------------------
#         # STEP 2: Lightweight Outcome Models mu_1(X) and mu_0(X)
#         # ----------------------------------------------------
#         treated_mask = W_train == 1
#         out_model_1 = RandomForestClassifier(
#             n_estimators=100, 
#             max_depth=10, 
#             min_samples_leaf=10,
#             random_state=42, 
#             n_jobs=-1
#         )
#         out_model_1.fit(X_train[treated_mask], Y_train[treated_mask])
#         mu_1_hat[val_idx] = out_model_1.predict_proba(X_val)[:, 1]

#         control_mask = W_train == 0
#         out_model_0 = RandomForestClassifier(
#             n_estimators=100, 
#             max_depth=10, 
#             min_samples_leaf=10,
#             random_state=42, 
#             n_jobs=-1
#         )
#         out_model_0.fit(X_train[control_mask], Y_train[control_mask])
#         mu_0_hat[val_idx] = out_model_0.predict_proba(X_val)[:, 1]

#     # ----------------------------------------------------
#     # STEP 3: Symmetric Trimming & AIPW Core Math
#     # ----------------------------------------------------
#     e_hat = np.clip(e_hat, 0.05, 0.95)

#     base_diff = mu_1_hat - mu_0_hat
#     treated_correction = (W * (Y - mu_1_hat)) / e_hat
#     control_correction = ((1 - W) * (Y - mu_0_hat)) / (1 - e_hat)

#     aipw_scores = base_diff + treated_correction - control_correction
#     ate = np.mean(aipw_scores)

#     # Asymptotic Standard Errors via Influence Curve
#     influence_curve = aipw_scores - ate
#     variance = np.var(influence_curve, ddof=1) / N
#     std_error = np.sqrt(variance)

#     # Inference Math
#     z_stat = ate / std_error
#     p_val = 2 * (1 - stats.norm.cdf(np.abs(z_stat)))
#     ci_lower = ate - (1.96 * std_error)
#     ci_upper = ate + (1.96 * std_error)

#     print(f"\nOutcome: {name}")
#     print(f"  ATE: {ate:+.4f} ({ate*100:+.2f} percentage points)")
#     print(f"  Std Error: {std_error:.4f}")
#     print(f"  95% CI:   [{ci_lower:+.4f}, {ci_upper:+.4f}]")
#     print(f"  p-value:   {p_val:.4f}")

--- RUNNING LIGHTWEIGHT RANDOM FOREST AIPW ---

Outcome: Voted for Biden
  ATE: -0.0462 (-4.62 percentage points)
  Std Error: 0.0194
  95% CI:   [-0.0842, -0.0081]
  p-value:   0.0174

Outcome: Voted for Trump
  ATE: +0.0469 (+4.69 percentage points)
  Std Error: 0.0192
  95% CI:   [+0.0093, +0.0846]
  p-value:   0.0146


In [179]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.ensemble import RandomForestClassifier

# Use our clean master X_matrix values
X = X_matrix.values
N = len(W)

candidate_outcomes = {"Voted for Biden": Y_biden, "Voted for Trump": Y_trump}

# Native NumPy K-Fold generation
np.random.seed(42)
indices = np.arange(N)
np.random.shuffle(indices)

num_folds = 5
folds = np.array_split(indices, num_folds)

print("--- RUNNING POPULATION-WEIGHTED RANDOM FOREST AIPW ---")

for name, Y in candidate_outcomes.items():
    # Placeholders for out-of-fold probability predictions
    e_hat = np.zeros(N)
    mu_1_hat = np.zeros(N)
    mu_0_hat = np.zeros(N)

    for f in range(num_folds):
        val_idx = folds[f]
        train_idx = np.setdiff1d(indices, val_idx)

        X_train, X_val = X[train_idx], X[val_idx]
        W_train, W_val = W[train_idx], W[val_idx]
        Y_train, Y_val = Y[train_idx], Y[val_idx]

        # Slice your attrition-repaired weights for this training fold
        w_train = survey_weights[train_idx]
        w_train = w_train / np.mean(w_train)  # Re-normalize locally for tree stability

        # ----------------------------------------------------
        # STEP 1: Population-Weighted Propensity Score Model e(X)
        # ----------------------------------------------------
        prop_model = RandomForestClassifier(
            n_estimators=100, 
            max_depth=10, 
            min_samples_leaf=10,
            random_state=42, 
            n_jobs=-1
        )
        # Pass the weights to force general population-aligned tree splits
        prop_model.fit(X_train, W_train, sample_weight=w_train)
        e_hat[val_idx] = prop_model.predict_proba(X_val)[:, 1]

        # ----------------------------------------------------
        # STEP 2: Population-Weighted Outcome Models mu_1(X) and mu_0(X)
        # ----------------------------------------------------
        # Fit model only on the treated units (W == 1)
        treated_mask = W_train == 1
        w_train_1 = w_train[treated_mask] / np.mean(w_train[treated_mask])  # Local re-centering
        
        out_model_1 = RandomForestClassifier(
            n_estimators=100, 
            max_depth=10, 
            min_samples_leaf=10,
            random_state=42, 
            n_jobs=-1
        )
        out_model_1.fit(X_train[treated_mask], Y_train[treated_mask], sample_weight=w_train_1)
        mu_1_hat[val_idx] = out_model_1.predict_proba(X_val)[:, 1]

        # Fit model only on the control units (W == 0)
        control_mask = W_train == 0
        w_train_0 = w_train[control_mask] / np.mean(w_train[control_mask])  # Local re-centering
        
        out_model_0 = RandomForestClassifier(
            n_estimators=100, 
            max_depth=10, 
            min_samples_leaf=10,
            random_state=42, 
            n_jobs=-1
        )
        out_model_0.fit(X_train[control_mask], Y_train[control_mask], sample_weight=w_train_0)
        mu_0_hat[val_idx] = out_model_0.predict_proba(X_val)[:, 1]

    # ----------------------------------------------------
    # STEP 3: Symmetric Trimming & Population-Weighted Math
    # ----------------------------------------------------
    e_hat = np.clip(e_hat, 0.05, 0.95)

    base_diff = mu_1_hat - mu_0_hat
    treated_correction = (W * (Y - mu_1_hat)) / e_hat
    control_correction = ((1 - W) * (Y - mu_0_hat)) / (1 - e_hat)

    # Generate point-level double-robust scores
    aipw_scores = base_diff + treated_correction - control_correction
    
    # TRUE POPULATION-WEIGHTED ATE
    population_ate = np.average(aipw_scores, weights=survey_weights)

    # Analytical Standard Errors via Weighted Empirical Influence Curve
    influence_curve = survey_weights * (aipw_scores - population_ate)
    variance = np.sum(influence_curve**2) / (N**2)
    std_error = np.sqrt(variance)

    # Inference Math
    z_stat = population_ate / std_error
    p_val = 2 * (1 - stats.norm.cdf(np.abs(z_stat)))
    ci_lower = population_ate - (1.96 * std_error)
    ci_upper = population_ate + (1.96 * std_error)

    print(f"\nOutcome: {name} (Population Weighted)")
    print(f"  ATE: {population_ate:+.4f} ({population_ate*100:+.2f} percentage points)")
    print(f"  Std Error: {std_error:.4f}")
    print(f"  95% CI:   [{ci_lower:+.4f}, {ci_upper:+.4f}]")
    print(f"  p-value:   {p_val:.4f}")

--- RUNNING POPULATION-WEIGHTED RANDOM FOREST AIPW ---

Outcome: Voted for Biden (Population Weighted)
  ATE: -0.0367 (-3.67 percentage points)
  Std Error: 0.0222
  95% CI:   [-0.0803, +0.0069]
  p-value:   0.0986

Outcome: Voted for Trump (Population Weighted)
  ATE: +0.0657 (+6.57 percentage points)
  Std Error: 0.0252
  95% CI:   [+0.0163, +0.1152]
  p-value:   0.0092


In [187]:
import numpy as np
import pandas as pd

print("--- POPULATION-WEIGHTED ATTRITION PROFILE (TARGET POPULATION SKEWS) ---")

# ==========================================
# FIX 1: APPLY FEATURE MAPPINGS TO df_raw
# ==========================================
# df_raw needs the engineered columns before we can profile them
df_raw["ideo5_linear"] = df_raw["ideo5"].map({"Very liberal": 1, "Liberal": 2, "Moderate": 3, "Conservative": 4, "Very conservative": 5})
df_raw["pid7_linear"] = df_raw["pid7"].map({"Strong Democrat": 1, "Not very strong Democrat": 2, "Lean Democrat": 3, "Independent": 4, "Lean Republican": 5, "Not very strong Republican": 6, "Strong Republican": 7})

socmed_mapping = {"Less than 10 minutes per day": 1, "10–30 minutes per day": 2, "31–60 minutes per day": 3, "1–2 hours per day": 4, "2–3 hours per day": 5, "More than 3 hours per day": 6}
df_raw["socmed_use_linear"] = df_raw["socmed_use"].astype(str).str.strip().map(socmed_mapping)

newsint_mapping = {"Hardly at all": 1, "Only now and then": 2, "Some of the time": 3, "Most of the time": 4}
df_raw["newsint_linear"] = df_raw["newsint"].astype(str).str.strip().map(newsint_mapping)

intuse_mapping = {"Less often": 1, "About once a day": 3, "Several times a day": 4, "A few times a week": 2}
df_raw["intuse_linear"] = df_raw["intuse"].astype(str).str.strip().map(intuse_mapping).fillna(3)

# Assuming condense_2016_vote_fixed is still loaded in your environment from the previous script
df_raw["presvote16post_condensed"] = df_raw["presvote16post"].apply(condense_2016_vote_fixed)

# ==========================================
# FIX 2: ALIGN MASKS FOR NUMPY ARRAY SLICING
# ==========================================
df_raw["survived_preprocessing"] = df_raw.index.isin(df_clean.index).astype(int)
raw_base_weights = df_raw["weight"].fillna(1.0).values

profile_features = [
    "age", "female", "ideo5_linear", "pid7_linear", 
    "socmed_use_linear", "newsint_linear", "intuse_linear"
]

categorical_profiles = [
    "race4", "educ4", "presvote16post_condensed"
]

weighted_profile_rows = []

# --- Part A: Weighted Means for Continuous / Linearized Features ---
for col in profile_features:
    # Cast pandas masks to .values to slice the numpy weight array cleanly
    stay_mask = ((df_raw["survived_preprocessing"] == 1) & df_raw[col].notna()).values
    drop_mask = ((df_raw["survived_preprocessing"] == 0) & df_raw[col].notna()).values
    
    # Calculate population-weighted averages
    mean_stay = np.average(df_raw.loc[stay_mask, col], weights=raw_base_weights[stay_mask])
    mean_drop = np.average(df_raw.loc[drop_mask, col], weights=raw_base_weights[drop_mask])
    
    # Weighted Standard Deviations for proper SMD
    variance_stay = np.average((df_raw.loc[stay_mask, col] - mean_stay)**2, weights=raw_base_weights[stay_mask])
    variance_drop = np.average((df_raw.loc[drop_mask, col] - mean_drop)**2, weights=raw_base_weights[drop_mask])
    pooled_sd = np.sqrt((variance_stay + variance_drop) / 2)
    
    smd = (mean_stay - mean_drop) / pooled_sd if pooled_sd > 0 else 0
    
    weighted_profile_rows.append({
        "Feature Variable": col,
        "Population Share (Stay)": f"{mean_stay:.2f}",
        "Population Share (Leave)": f"{mean_drop:.2f}",
        "Std. Difference (SMD)": f"{smd:+.3f}"
    })

# --- Part B: Weighted Proportions for Categorical Features ---
for col in categorical_profiles:
    valid_mask = df_raw[col].notna()
    unique_cats = sorted(df_raw.loc[valid_mask, col].unique())
    
    for cat in unique_cats:
        stay_mask = ((df_raw["survived_preprocessing"] == 1) & (df_raw[col] == cat)).values
        drop_mask = ((df_raw["survived_preprocessing"] == 0) & (df_raw[col] == cat)).values
        
        # Use .values on compound masks for safe summation
        valid_stay_mask = ((df_raw["survived_preprocessing"] == 1) & valid_mask).values
        valid_drop_mask = ((df_raw["survived_preprocessing"] == 0) & valid_mask).values
        
        total_w_stay = raw_base_weights[valid_stay_mask].sum()
        total_w_drop = raw_base_weights[valid_drop_mask].sum()
        
        p_stay = raw_base_weights[stay_mask].sum() / total_w_stay if total_w_stay > 0 else 0
        p_drop = raw_base_weights[drop_mask].sum() / total_w_drop if total_w_drop > 0 else 0
        
        # SMD for proportions
        pooled_sd = np.sqrt((p_stay * (1 - p_stay) + p_drop * (1 - p_drop)) / 2)
        smd = (p_stay - p_drop) / pooled_sd if pooled_sd > 0 else 0
        
        weighted_profile_rows.append({
            "Feature Variable": f"{col} -> {cat}",
            "Population Share (Stay)": f"{p_stay * 100:.1f}%",
            "Population Share (Leave)": f"{p_drop * 100:.1f}%",
            "Std. Difference (SMD)": f"{smd:+.3f}"
        })

# Render Table
df_weighted_profile = pd.DataFrame(weighted_profile_rows)
print(df_weighted_profile.to_string(index=False))

--- POPULATION-WEIGHTED ATTRITION PROFILE (TARGET POPULATION SKEWS) ---
                           Feature Variable Population Share (Stay) Population Share (Leave) Std. Difference (SMD)
                                        age                   52.03                    49.52                +0.155
                                     female                    0.50                     0.61                -0.221
                               ideo5_linear                    3.05                     3.00                +0.044
                                pid7_linear                    3.70                     4.04                -0.146
                          socmed_use_linear                    2.57                     2.81                -0.149
                             newsint_linear                    3.00                     2.70                +0.331
                              intuse_linear                    3.83                     3.81                +0.051
        

In [160]:
df_raw["weight_post"]

0       1.7017
1       0.2715
2       0.9867
3       0.7919
4       1.0359
         ...  
1146    0.3385
1147    0.4097
1148    4.0906
1149    0.2818
1150    1.1531
Name: weight_post, Length: 1151, dtype: float64

In [162]:
df_raw["weight"]

0       0.717401
1       0.376286
2       3.584190
3       1.018029
4       0.388803
          ...   
1146    1.178774
1147    0.278651
1148    4.472646
1149    0.282634
1150    2.490094
Name: weight, Length: 1151, dtype: float64